In [12]:
import importlib
import pymc as pm
import pytensor.tensor as pt
from pathlib import Path
import numpy as np 
from src.scripts.get_matrix_and_specs import get_respmatrix, get_meas_spec
from src.scripts.paths import ProjectPaths_Binding as PP

scripts_dir = PP.SRC_DIR


def get_cached_data(name, low, high, bin_width, compute_func):
    cache_dir = Path(f"{scripts_dir}/../../gen/cache_files")
    cache_dir.mkdir(exist_ok=True)
    
    file_path = cache_dir / f"{name}_{low}_{high}_bw{bin_width}.npy"
    
    if file_path.exists():
        return np.load(file_path)
    
    print(f"Calculate {name} initial (Cache: {low}-{high} keV, BW: {bin_width})...")
    data = compute_func()
    np.save(file_path, data)
    return data


def get_rema_100M(casc: str, det: str, mask: np.ndarray, bin_width: int)-> np.ndarray: 
    return (get_respmatrix('100M', casc, det, bin_width)[np.ix_(mask, mask)])

def pytensor_interp(x_new, x_old, y_old):
    return pt.extra_ops.interp(x_new, x_old, y_old)


class binding:
    def __init__(self, 
                 project: str,
                 runnr: int,  
                 savename: str, 
                 bin_width: int = 10, 
                 max_energy = 10000,
                 branchings = True, 
                 br12p2 = False,
                 O_cont = False,
                 ng_cont = False):
        
        self.O_cont_bool = O_cont
        self.ng_cont_bool = ng_cont
        self.branch_bool = branchings
        self.br1_2p2_bool = br12p2

        deconv_params = importlib.import_module(f'src.projects.{project}.deconv_params').deconv_params



        try: 
            self.runnr = int(runnr)
        except:
            self.runnr = str(runnr)

        
        self.savename = savename
        self.ebeam = deconv_params[runnr]['beam']
        self.bin_width = bin_width
        self.bin_centers = np.arange(bin_width, max_energy + bin_width, bin_width)
        self.low = deconv_params[runnr]['low']
        self.high = deconv_params[runnr]['high']
        self.mask_energy = (self.bin_centers >= self.low) & (self.bin_centers <= self.high) 
        self.td_cut_low = deconv_params[runnr]['cut_dets_low']# [L1, L3, 5, 7]
        self.td_cut_high = deconv_params[runnr]['cut_dets_high']





binding('70Zn_2024', 179, 'bins' )
